## Récupération des données annotées

In [1]:
import pandas as pd
import pickle

In [2]:
# Fichier contenant les phrases segmentées
df_phrases = pd.read_csv("./artifacts/bertopic/reviews_phrases_with_topics.csv")

# Fichier clusters annotés
df_clusters = pd.read_excel("./artifacts/bertopic/topics_top_words_phrases_annoté.xlsx")

# Normalisation des catégories
replace_map = {
    "Qualité Produit": "qualité produit",
    "Qualité produit": "qualité produit",
    "Service Livraison": "service livraison",
    "Service livraison": "service livraison",
    "Service Client": "service client",
    "Service client": "service client",
}

def normalize_category(cat):
    if pd.isna(cat):
        return None
    cat = cat.strip().lower()
    return replace_map.get(cat, cat)

df_clusters["Catégorie"] = df_clusters["Catégorie"].apply(normalize_category)

# Fusion sur le numéro de topic
df_phrases_merged = df_phrases.merge(
    df_clusters[["Topic", "Catégorie"]],
    left_on='topics',
    right_on='Topic',
    how='left'
).drop(columns=['Topic'])

# Catégories possibles
categories = ["qualité produit", "service livraison", "service client"]

# Retirer les phrases non annotés ou avec plusieurs catégories
df_phrases_merged = df_phrases_merged[df_phrases_merged['Catégorie'].isin(categories)]


# Création colonnes one-hot binaires
for cat in categories:
    df_phrases_merged[cat] = (df_phrases_merged['Catégorie'] == cat).astype(int)

df_phrases.to_csv("./../resultats/bertopic/data/dataset_phrases.csv", index=False)

In [3]:
cols_invariantes = ["Commentaire", "star", "date", "client", "reponse", "source", "company", "ville", "maj", "date_commande", "ecart", "clean_comment"]

agg_dict = {}

# Colonnes invariantes : first
for col in cols_invariantes:
    agg_dict[col] = (col, "first")

# Colonnes de labels : max
for col in categories:
    agg_dict[col] = (col, "max")

# Aggrégation pour avoir les catégories par avis
df_avis = (
    df_phrases_merged
    .groupby("comment_id", as_index=False)
    .agg(**agg_dict)
)

df_avis.to_csv("./../resultats/bertopic/data/dataset_avis.csv", index=False)

In [4]:
df_phrases_merged.head()

,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,comment_id,sentence,topics,Catégorie,qualité produit,service livraison,service client
16,"Vente Lacoste Honteuse , article erroné , arti...",1,2021-06-19 00:00:00+00:00,Vanessa L,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,"vente lacoste honteuse , article erroné , arti...",2,( commande n°230077467 et commande n•230077467...,97,qualité produit,1,0,0
24,Annulation de commande après 2 mois d ’ attent...,1,2021-06-18 00:00:00+00:00,aurore regnier,"Bonjour Aurore , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,annulation de commande après 2 mois d ’ attent...,6,annulation de commande après 2 mois d ’ attent...,62,service client,0,0,1
32,Extrêmement deçu pour mes achats lors la vente...,1,2021-06-18 00:00:00+00:00,Ayna,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,extrêmement deçu pour mes achats lors la vente...,8,extrêmement deçu pour mes achats lors la vente...,87,qualité produit,1,0,0
33,S'il y'avait une option : ne pas mettre d'étoi...,1,2021-06-18 00:00:00+00:00,linda Ng,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,s'il y'avait une option : ne pas mettre d'étoi...,9,s'il y'avait une option : ne pas mettre d'étoi...,134,qualité produit,1,0,0
40,ARNAQUE J ’ ai acheté une combinaison blanche ...,1,2021-06-18 00:00:00+00:00,Sarah,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,arnaque j ’ ai acheté une combinaison blanche ...,10,grosse arnaque c ’ est inadmissible je vais la...,87,qualité produit,1,0,0


In [5]:
df_avis.head()

,comment_id,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,qualité produit,service livraison,service client
0,2,"Vente Lacoste Honteuse , article erroné , arti...",1,2021-06-19 00:00:00+00:00,Vanessa L,None,TrustPilot,ShowRoom,None,None,None,NaN,"vente lacoste honteuse , article erroné , arti...",1,0,0
1,6,Annulation de commande après 2 mois d ’ attent...,1,2021-06-18 00:00:00+00:00,aurore regnier,"Bonjour Aurore , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,None,None,None,NaN,annulation de commande après 2 mois d ’ attent...,0,0,1
2,8,Extrêmement deçu pour mes achats lors la vente...,1,2021-06-18 00:00:00+00:00,Ayna,None,TrustPilot,ShowRoom,None,None,None,NaN,extrêmement deçu pour mes achats lors la vente...,1,0,0
3,9,S'il y'avait une option : ne pas mettre d'étoi...,1,2021-06-18 00:00:00+00:00,linda Ng,None,TrustPilot,ShowRoom,None,None,None,NaN,s'il y'avait une option : ne pas mettre d'étoi...,1,0,0
4,10,ARNAQUE J ’ ai acheté une combinaison blanche ...,1,2021-06-18 00:00:00+00:00,Sarah,None,TrustPilot,ShowRoom,None,None,None,NaN,arnaque j ’ ai acheté une combinaison blanche ...,1,0,0


## XGBoost

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics import f1_score

/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df_avis = pd.read_csv('./../data/processed/dataset_avis.csv')

In [8]:
TEXT_COL = "clean_comment"
LABEL_COLS = ["qualité produit", "service livraison", "service client"]

label_names = LABEL_COLS
num_labels = len(label_names)
print(label_names)

['qualité produit', 'service livraison', 'service client']


In [9]:
from sklearn.model_selection import KFold, train_test_split
import numpy as np

texts = df_avis[TEXT_COL].values
labels = df_avis[LABEL_COLS].values

texts, X_test, labels, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

#### Sélection des hyperparamètres

In [10]:
embedding_models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "camembert-base",
    "paraphrase-multilingual-MiniLM-L12-v2"
]

max_depths = [3, 4, 5]

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [11]:
f1_micro_scores = []
f1_weighted_scores = []

for emb_name in embedding_models:
    print(f"\nEmbedding : {emb_name}")
    embedder = SentenceTransformer(emb_name)

    for max_depth in max_depths:
        print(f"max_depth = {max_depth}")

        f1_micro_scores = []
        f1_weighted_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(texts), 1):

            # Split
            X_train_texts = texts[train_idx]
            X_val_texts = texts[val_idx]
            y_train = labels[train_idx]
            y_val = labels[val_idx]

            # Embeddings
            X_train = embedder.encode(X_train_texts, convert_to_numpy=True, show_progress_bar=False)
            X_val = embedder.encode(X_val_texts, convert_to_numpy=True, show_progress_bar=False)

            # Modèle
            model = MultiOutputClassifier(
                XGBClassifier(
                    eval_metric="logloss",
                    n_estimators=100,
                    max_depth=max_depth,
                    learning_rate=0.1,
                    random_state=42
                )
            )

            # Entraînement
            model.fit(X_train, y_train)

            # Prédiction
            y_pred = model.predict(X_val)

            # Scores
            f1_micro_scores.append(
                f1_score(y_val, y_pred, average="micro")
            )
            f1_weighted_scores.append(
                f1_score(y_val, y_pred, average="weighted")
            )

        print(
            f"    F1 micro: {np.mean(f1_micro_scores):.4f} | "
            f"F1 weighted: {np.mean(f1_weighted_scores):.4f}"
        )



Embedding : all-MiniLM-L6-v2
max_depth = 3
    F1 micro: 0.6955 | F1 weighted: 0.6734
max_depth = 4
    F1 micro: 0.6993 | F1 weighted: 0.6797
max_depth = 5
    F1 micro: 0.7031 | F1 weighted: 0.6825

Embedding : all-mpnet-base-v2
max_depth = 3
    F1 micro: 0.6914 | F1 weighted: 0.6704
max_depth = 4
    F1 micro: 0.6976 | F1 weighted: 0.6787
max_depth = 5
    F1 micro: 0.6984 | F1 weighted: 0.6779

Embedding : camembert-base


No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.


max_depth = 3
    F1 micro: 0.7348 | F1 weighted: 0.7214
max_depth = 4
    F1 micro: 0.7367 | F1 weighted: 0.7237
max_depth = 5
    F1 micro: 0.7369 | F1 weighted: 0.7234

Embedding : paraphrase-multilingual-MiniLM-L12-v2
max_depth = 3
    F1 micro: 0.7022 | F1 weighted: 0.6857
max_depth = 4
    F1 micro: 0.7083 | F1 weighted: 0.6923
max_depth = 5
    F1 micro: 0.7060 | F1 weighted: 0.6885


#### Entraînement final

In [12]:
embedder = SentenceTransformer('camembert-base')
max_depth = 5

X_train = embedder.encode(texts, convert_to_numpy=True)
X_test = embedder.encode(X_test, convert_to_numpy=True)

No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.


In [13]:
# Modèle
model = MultiOutputClassifier(
    XGBClassifier(
        eval_metric="logloss",
        n_estimators=100,
        max_depth=max_depth,
        learning_rate=0.1,
        random_state=42
    )
)

# Entraînement
model.fit(X_train, labels)

# Prédictions
y_pred = model.predict(X_test)

# Scores par label
for i, col in enumerate(LABEL_COLS):
    acc = np.mean(y_test[:, i] == y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores
f1_micro = f1_score(y_test, y_pred, average="micro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"F1 micro     : {f1_micro:.4f}")
print(f"F1 weighted  : {f1_weighted:.4f}")

Label 'qualité produit': Accuracy = 0.786, F1-score = 0.816
Label 'service livraison': Accuracy = 0.793, F1-score = 0.731
Label 'service client': Accuracy = 0.866, F1-score = 0.403
F1 micro     : 0.7430
F1 weighted  : 0.7260


#### Evaluation sur le dataset de test

In [14]:
from sklearn.metrics import f1_score
import numpy as np

# Prédiction sur le test set
y_pred = model.predict(X_test)

# Scores par label
for i, col in enumerate(LABEL_COLS):
    acc = np.mean(y_test[:, i] == y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores globaux
f1_micro = f1_score(y_test, y_pred, average='micro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"F1-score micro    : {f1_micro:.3f}")
print(f"F1-score weighted : {f1_weighted:.3f}")

Label 'qualité produit': Accuracy = 0.786, F1-score = 0.816
Label 'service livraison': Accuracy = 0.793, F1-score = 0.731
Label 'service client': Accuracy = 0.866, F1-score = 0.403
F1-score micro    : 0.743
F1-score weighted : 0.726


In [15]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.77      0.87      0.82       717
service livraison       0.80      0.67      0.73       547
   service client       0.70      0.28      0.40       209

        micro avg       0.78      0.71      0.74      1473
        macro avg       0.76      0.61      0.65      1473
     weighted avg       0.77      0.71      0.73      1473
      samples avg       0.74      0.74      0.73      1473



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [17]:
from sklearn.metrics import confusion_matrix

for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(y_test[:, i], y_pred[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,405,185
Vrai 1,95,622


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,670,90
Vrai 1,180,367


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,1073,25
Vrai 1,150,59


#### Evaluation sur le dataset gold annoté manuellement

In [36]:
df_avis_gold = pd.read_csv('./../data/test_dataset/100_avis_annote.csv', sep=";")
X_gold = df_avis_gold[TEXT_COL].values
y_gold = df_avis_gold[LABEL_COLS].values
X_gold = embedder.encode(X_gold, convert_to_numpy=True)

In [37]:
from sklearn.metrics import f1_score
import numpy as np

# Prédiction sur le test set
y_pred = model.predict(X_gold)

# Scores par label
for i, col in enumerate(LABEL_COLS):
    acc = np.mean(y_gold[:, i] == y_pred[:, i])
    f1 = f1_score(y_gold[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores globaux
f1_micro = f1_score(y_gold, y_pred, average='micro')
f1_weighted = f1_score(y_gold, y_pred, average='weighted')

print(f"F1-score micro    : {f1_micro:.3f}")
print(f"F1-score weighted : {f1_weighted:.3f}")

Label 'qualité produit': Accuracy = 0.650, F1-score = 0.646
Label 'service livraison': Accuracy = 0.820, F1-score = 0.591
Label 'service client': Accuracy = 0.720, F1-score = 0.067
F1-score micro    : 0.532
F1-score weighted : 0.457


In [39]:
from sklearn.metrics import classification_report

print(classification_report(y_gold, y_pred, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.55      0.78      0.65        41
service livraison       0.57      0.62      0.59        21
   service client       0.33      0.04      0.07        27

        micro avg       0.55      0.52      0.53        89
        macro avg       0.48      0.48      0.43        89
     weighted avg       0.49      0.52      0.46        89
      samples avg       0.45      0.41      0.42        89



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-define

In [40]:
from sklearn.metrics import confusion_matrix

for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(y_gold[:, i], y_pred[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,33,26
Vrai 1,9,32


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,69,10
Vrai 1,8,13


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,71,2
Vrai 1,26,1
